## tl;dr
The 000973 common mesh RGB base is rejected: 0/3 held-view visual passes. Face PSNR improves, but LPIPS does not, and the neck strip becomes more conspicuous. This companion validates the retained experiment; it does not fit on held-out RGB.

## Context & Methods
As of 2026-09-06 UTC. Same fixed mesh, calibration, visibility, hard labels and study face ROI. Only `source RGB - source base + common base` changes. All 62 train cameras fit the mesh bases.

### Key Assumptions
Face-only metrics exclude neck, tube, hand and ear. Native visual review therefore governs acceptance. This repeatedly inspected frame is diagnostic, not untouched temporal/fly-through generalization. No comparison to the old campaign ROI is valid. Use the repository `.venv` plus the recorded private dependency path.

## Data
Local source of truth: `canonical_surface_base_control/findings.json`, its integrity audit and per-view manifests. The independent audit script is retained under `/mnt/data/lookcloser_dec5_5a3_surface_repair/tools/audit_canonical_surface_base.py`.

In [1]:
from pathlib import Path
import hashlib, json, os
os.environ.setdefault('OPENCV_IO_ENABLE_OPENEXR', '1')
import numpy as np
from PIL import Image, ImageDraw
import cv2
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/canonical_surface_base_control')
findings = json.loads((root/'findings.json').read_text())
audit = json.loads((root/'integrity_audit.json').read_text())
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()
for path, expected in audit['hashes'].items():
    assert digest(path) == expected, path
assert len(audit['hashes']) == audit['hash_count']
print({'verified_hashes': audit['hash_count'], 'views': audit['view_inventory']})

{'verified_hashes': 157, 'views': ['F', 'J', 'L']}


## Results
### 1. Recompute exact face-pixel PSNR
The scorer uses the same GT-only polygon for both predictions. SSIM/LPIPS values are retained scorer outputs, with identical protocol; PSNR is independently recomputed below using float64 NumPy.

In [2]:
metric_rows = []
for variant in ['off', 'common']:
    m = json.loads((root/f'metrics_{variant}/metrics.json').read_text())
    roi = json.loads(Path(m['face_polygons']).read_text())
    canvas = Image.new('L', (1920,1080)); draw = ImageDraw.Draw(canvas)
    for polygon in roi['include_polygons']:
        draw.polygon([tuple(p) for p in polygon], fill=255)
    for polygon in roi['exclude_polygons']:
        draw.polygon([tuple(p) for p in polygon], fill=0)
    mask = np.asarray(canvas) > 0
    gt = cv2.imread(m['ground_truth'], cv2.IMREAD_UNCHANGED)[...,::-1].clip(0,1).astype(np.float64)
    pred = np.asarray(Image.open(m['prediction']), dtype=np.float64)/255
    psnr = float(-10*np.log10(np.mean((pred[mask]-gt[mask])**2)))
    assert abs(psnr-m['face_psnr']) < 1e-5
    assert m['protocol']['candidate_surface_mask'] is False
    metric_rows.append({'variant':variant, 'face_psnr':psnr, 'face_ssim':m['face_ssim'], 'face_lpips':m['face_lpips']})
metric_rows

[{'variant': 'off',
  'face_psnr': 28.724287111269923,
  'face_ssim': 0.889098048210144,
  'face_lpips': 0.04798922687768936},
 {'variant': 'common',
  'face_psnr': 29.360822354135255,
  'face_ssim': 0.8888427019119263,
  'face_lpips': 0.048490166664123535}]

### 2. Check replay, support and pointwise color change
All source-label images are identical between off/on. Added color offsets must explain the float render exactly. Pre-existing point probes illustrate the new darkening; they are not fitting targets or aggregate metrics.

In [3]:
original = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/depth_separated_source_control/off_rank001')
path_manifest = json.loads((original/'path_manifest.json').read_text())
for i, view in enumerate('FJL'):
    folder = root/f'view_{view}'
    assert digest(folder/'off/eval_pred_0000.png') == path_manifest['views'][i]['sha256']
    assert digest(folder/'off/source_selection.png') == digest(folder/'common_base/source_selection.png')
    raw = cv2.imread(str(original/f'scratch/{i:04d}/render/seam_cut8/eval_pred_0000.exr'), cv2.IMREAD_UNCHANGED)[...,::-1]
    after = cv2.imread(str(folder/'common_base/eval_pred_0000.exr'), cv2.IMREAD_UNCHANGED)[...,::-1]
    with np.load(folder/'applied_offsets.npz') as z:
        offsets = z['offsets']
    assert np.isfinite(after).all()
    np.testing.assert_array_equal(after, (raw+offsets).clip(0,1))
review = json.loads((root/'visual_review.json').read_text())
assert [v['status'] for v in review['views']] == ['fail']*3
[{'xy':[p['x'],p['y']], 'before':p['before_rgb8'], 'after':p['after_rgb8'], 'GT_posthoc':p['gt_rgb8']} for p in findings['paired_point_checks']]

[{'xy': [580, 648],
  'before': [152, 116, 78],
  'after': [137, 106, 71],
  'GT_posthoc': [151, 115, 83]},
 {'xy': [580, 649],
  'before': [156, 119, 86],
  'after': [141, 107, 78],
  'GT_posthoc': [151, 115, 83]},
 {'xy': [580, 660],
  'before': [168, 125, 87],
  'after': [169, 125, 88],
  'GT_posthoc': [160, 117, 83]}]

## Takeaways
The explicit common base does not solve this patch. At `(580,648)` it darkens `[152,116,78]` to `[137,106,71]` (GT `[151,115,83]`); the previous interior probe barely changes. This isolates the added darkening to color replacement, not geometry mutation. It does not prove exposure is the original disagreement source. All seven native detail crops plus overview were separately viewed and are linked in the Markdown report/visual verdict. Original campaign and defaults are preserved; no temporal or fly-through pass is claimed.